In [1]:
!nvidia-smi

Tue Aug 25 00:34:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install nvcc4jupyter

In [3]:
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpysrm363l".


In [19]:
%%cuda
#include <iostream>

// This is the "Kernel" - it runs directly on the T4 GPU cores
__global__ void helloFromGPU() {
    auto threadId = threadIdx.x;
    auto blockId = blockIdx.x;
    printf("Hello from T4 GPU! Block: %d, Thread: %d\n", blockId, threadId);
}

int main() {
    std::cout << "Hello from the CPU!" << std::endl;

    // Launch the function on the GPU: 2 blocks, 4 threads per block (8 parallel threads total)
    helloFromGPU<<<2, 4>>>();

    // Tell the CPU to wait for the GPU to finish printing before exiting
    cudaDeviceSynchronize();

    return 0;
}

Hello from the CPU!
Hello from T4 GPU! Block: 0, Thread: 0
Hello from T4 GPU! Block: 0, Thread: 1
Hello from T4 GPU! Block: 0, Thread: 2
Hello from T4 GPU! Block: 0, Thread: 3
Hello from T4 GPU! Block: 1, Thread: 0
Hello from T4 GPU! Block: 1, Thread: 1
Hello from T4 GPU! Block: 1, Thread: 2
Hello from T4 GPU! Block: 1, Thread: 3



In [15]:
%%cuda
#include <iostream>
#include <chrono>
#include <cstdint>
#include <algorithm>
#include <cmath>

bool check(float* A, float* B, float* C, uint64_t n)
{
    for (uint64_t i = 0; i < n; i++)
    {
        if (C[i] != (A[i] + B[i]))
        {
            std::cout << "ERROR at idx: " << i << "Values: " << A[i] << ", " << B[i] <<  ", " << C[i];
            return false;
        }
    }
    std::cout << "Check: Success\n" << std::endl;
    return true;
}

void cpuVecAdd(float* A_h, float* B_h, float* C_h, uint64_t n)
{
    for (uint64_t i = 0; i < n; i++)
    {
        C_h[i] = A_h[i] + B_h[i];
    }
}

__global__ void vecAddKernel(float* A, float* B, float* C, uint64_t n)
{
    uint64_t i = threadIdx.x + blockIdx.x * blockDim.x;

    if (i < n)
        C[i] = A[i] + B[i];
}

void cudaVecAdd(float* A_h, float* B_h, float* C_h, uint64_t n)
{
    uint64_t size = n * sizeof(float);
    float *A_d, *B_d, *C_d;

    cudaMalloc((void**) &A_d, size);
    cudaMalloc((void**) &B_d, size);
    cudaMalloc((void**) &C_d, size);

    cudaMemcpy(A_d, A_h, size, cudaMemcpyHostToDevice);
    cudaMemcpy(B_d, B_h, size, cudaMemcpyHostToDevice);

    vecAddKernel<<<std::ceil(n/128.0), 128>>>(A_d, B_d, C_d, n);
    
    cudaMemcpy(C_h, C_d, size, cudaMemcpyDeviceToHost);
    
    cudaFree(A_d);    
    cudaFree(B_d);    
    cudaFree(C_d);    
}

int main()
{
    float *A, *B, *C;   
    uint64_t max = 5120000;

    for (uint64_t N = 10000; N <= max; N += 10000)
    {
        uint64_t size = N * sizeof(float); // size in bytes
        std::cout << "N: "  << N << " Size: " << size << std::endl;

        A = (float*)malloc(size);
        if(!A)
        {    
            std::cout << "ERROR IN A: Malloc failed" << std::endl;
            return -1;
        }

        B = (float*)malloc(size);
        if(!B)
        {    
            std::cout << "ERROR IN B: Malloc failed" << std::endl;
            return -1;
        }

        C = (float*)malloc(size);
        if(!C)
        {    
            std::cout << "ERROR IN C: Malloc failed" << std::endl;
            return -1;
        }    

        std::cout << "Before For Loop: " << std::endl;

        for (int i = 0; i < N; i++)
        {
            A[i] = i;
            B[i] = 2 * i;
        }
        
        auto t0 = std::chrono::steady_clock::now();
        cpuVecAdd(A,B,C,N);
        auto t1 = std::chrono::steady_clock::now();
        std::chrono::duration<double, std::milli> ms = t1 - t0;
        std::cout << "Cpu-VecAdd: " << ms.count() << " ms" << std::endl;

        auto t3 = std::chrono::steady_clock::now();
        cudaVecAdd(A,B,C,N);
        auto t4 = std::chrono::steady_clock::now();
        std::chrono::duration<double, std::milli> cudaMs = t4 - t3;
        std::cout << "Cuda-VecAdd: " << cudaMs.count() << " ms" << std::endl;

        check(A, B, C, N);

        if(A)
            free(A);
        if(B)
            free(B);
        if(C)
            free(C);
    }
}

N: 10000 Size: 40000
Before For Loop: 
Cpu-VecAdd: 0.042943 ms
Cuda-VecAdd: 291.777 ms
Check: Success

N: 20000 Size: 80000
Before For Loop: 
Cpu-VecAdd: 0.086135 ms
Cuda-VecAdd: 0.308027 ms
Check: Success

N: 30000 Size: 120000
Before For Loop: 
Cpu-VecAdd: 0.128488 ms
Cuda-VecAdd: 0.334048 ms
Check: Success

N: 40000 Size: 160000
Before For Loop: 
Cpu-VecAdd: 0.19437 ms
Cuda-VecAdd: 0.362793 ms
Check: Success

N: 50000 Size: 200000
Before For Loop: 
Cpu-VecAdd: 0.190454 ms
Cuda-VecAdd: 0.39921 ms
Check: Success

N: 60000 Size: 240000
Before For Loop: 
Cpu-VecAdd: 0.261202 ms
Cuda-VecAdd: 0.433963 ms
Check: Success

N: 70000 Size: 280000
Before For Loop: 
Cpu-VecAdd: 0.251742 ms
Cuda-VecAdd: 0.471903 ms
Check: Success

N: 80000 Size: 320000
Before For Loop: 
Cpu-VecAdd: 0.279447 ms
Cuda-VecAdd: 0.512189 ms
Check: Success

N: 90000 Size: 360000
Before For Loop: 
Cpu-VecAdd: 0.31669 ms
Cuda-VecAdd: 0.533762 ms
Check: Success

N: 100000 Size: 400000
Before For Loop: 
Cpu-VecAdd: 0.332698